In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!java --version

openjdk 11.0.28 2025-07-15
OpenJDK Runtime Environment (build 11.0.28+6-post-Ubuntu-1ubuntu122.04.1)
OpenJDK 64-Bit Server VM (build 11.0.28+6-post-Ubuntu-1ubuntu122.04.1, mixed mode, sharing)


In [3]:
!pip list | grep pyspark

pyspark                               3.5.1


In [4]:
# Remove old Java
!apt-get remove openjdk-* -y

# Install OpenJDK 17
!apt-get update -q
!apt-get install openjdk-17-jdk -y

# Set JAVA_HOME
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

# Verify
!java -version

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
Note, selecting 'openjdk-11-jdk' for glob 'openjdk-*'
Note, selecting 'openjdk-11-jre' for glob 'openjdk-*'
Note, selecting 'openjdk-11-jre-headless' for glob 'openjdk-*'
Note, selecting 'openjdk-19-jre-headless' for glob 'openjdk-*'
Note, selecting 'openjdk-8-jre-zero' for glob 'openjdk-*'
Note, selecting 'openjdk-11-jdk-headless' for glob 'openjdk-*'
Note, selecting 'openjdk-21-jdk-headless' for glob 'openjdk-*'
Note, selecting 'openjdk-19-jdk-headless' for glob 'openjdk-*'
Note, selecting 'openjdk-21-demo' for glob 'openjdk-*'
Note, selecting 'openjdk-18-jdk-headless' for glob 'openjdk-*'
Note, selecting 'openjdk-17-dbg' for glob 'openjdk-*'
Note, selecting 'openjdk-17-doc' for glob 'openjdk-*'
Note, selecting 'openjdk-18-dbg' for glob 'openjdk-*'
Note, selecting 'openjdk-17-jdk' for glob 'openjdk-*'
Note, selecting 'openjdk-18-doc' for glob 'openjdk-*'
Note, selecting 'openjdk-17-jre' f

In [5]:
!pip install --upgrade pyspark

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 434.2/434.2 MB 3.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.0/203.0 kB 17.0 MB/s eta 0:00:00
  Created wheel for pyspark: filename=pyspark-4.0.1-py2.py3-none-any.whl size=434813800 sha256=53115085e3ba24d432644dbee5a5ee7eea00a4147edc67142d5a4e0cc5cb5425
  Stored in directory: /root/.cache/pip/wheels/31/9f/68/f89fb34ccd886909be7d0e390eaaf97f21efdf540c0ee8dbcd
Successfully built pyspark
  Attempting uninstall: py4j
    Found existing installation: py4j 0.10.9.7
    Uninstalling py4j-0.10.9.7:
      Successfully uninstalled py4j-0.10.9.7
  Attempting uninstall: pyspark
    Found existing installation: pyspark 3.5.1
    Uninstalling pyspark-3.5.1:
      Successfully uninstalled pyspark-3.5.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-conn

## Dados

**Descrição das colunas:**  
timestamp,user_id,action,adId,campaignId

**Amostra:**  
2016-09-21 22:11:00,7c74953c-66cc-48bd-9d02-a02bf039cf3f,click,adId_09,campaignId_01  
2016-06-25 18:29:00,676a083e-2f8e-4ff2-9ec2-270f7f9d6033,view,adId_09,campaignId_02  
2016-02-14 19:03:00,77158997-0dfa-48b7-9149-973dc151ef8d,click,adId_02,campaignId_02  
2016-03-26 06:27:00,78aa2467-b502-413b-94e9-04ec8210bd13,click,adId_07,campaignId_03

**Path:**  
data/ad_action.csv

## Atividade 1

Qual é o anúncio mais popular, ou seja, com mais eventos independente do action?

**Resposta:**  
adId_06

## Atividade 2

Quantos clicks gerou a campanha mais popular?

**Resposta:**  
63983 clicks

## Atividade 3

Algum usuário só visualizou? Quantas actions foram enviadas por usuário que só visualizou?

**Resposta:**  
Sim, o usuário d99871cb-98b7-4ac5-97a5-b9a26c0f897b enviou apenas 1 action

## Atividade 4

Dos 10411 usuários, quantos usuários clicam mais que visualizam?

**Resposta:**  
10078

## Atividade 5

Calcule a quantidade de clicks por dia da semana e apresente o resultado em ordem decrescente de quantidade de clicks.

PS: Nesse exemplo, 0 é segunda-feira e 6 é domingo. Você pode usar outro padrão, contanto que os dias da semana estejam correspondentes a resposta abaixo

**Resposta**  
27918, 4  
25424, 5  
25028, 0  
25027, 3  
25020, 6  
24973, 1  
24915, 2

## Atividade 6

Algum anúncio teve um crescimento de clicks ao longo das 3 campanhas?

**Resposta:**  
Sim, o anúncio adId_07

In [6]:
# IMPORTS, CRIAÇÃO DO CLUSTER E CARREGAMENTO DOS DADOS AQUI
import numpy as np
import os
import pandas as pd
import pyspark.sql.functions as F

from pyspark.sql import SparkSession

os.environ['PYSPARK_SUBMIT_ARGS'] = '\
    --driver-memory 4G \
    --executor-memory 4G \
    pyspark-shell'

In [7]:
# Criando um cluster local com 1 executor e a quantidade de threads igual a quantidade de cores de CPU disponíveis

spark = SparkSession.builder\
    .master("local[*]")\
    .getOrCreate()
spark

In [8]:
# Comando para desativar os recursos do spark
# spark.stop()

In [9]:
AD_ACTION_CSV_PATH = 'drive/MyDrive/data/ad_action.csv'

In [10]:
df = spark.read.csv(AD_ACTION_CSV_PATH, header=False, inferSchema=True)\
  .toDF('timestamp', 'user_id', 'action', 'adId', 'campaignId')
df.show()

+-------------------+--------------------+------+-------+-------------+
|          timestamp|             user_id|action|   adId|   campaignId|
+-------------------+--------------------+------+-------+-------------+
|2016-09-21 22:11:00|7c74953c-66cc-48b...| click|adId_09|campaignId_01|
|2016-06-25 18:29:00|676a083e-2f8e-4ff...|  view|adId_09|campaignId_02|
|2016-02-14 19:03:00|77158997-0dfa-48b...| click|adId_02|campaignId_02|
|2016-03-26 06:27:00|78aa2467-b502-413...| click|adId_07|campaignId_03|
|2016-01-02 04:57:00|fef9a98c-d73e-48e...|  view|adId_02|campaignId_02|
|2016-03-04 09:14:00|6ba65af9-4d83-456...|  view|adId_09|campaignId_01|
|2016-07-09 21:42:00|be3befb9-ee08-431...| click|adId_09|campaignId_01|
|2016-07-18 21:15:00|d37832ae-546f-4b9...| click|adId_05|campaignId_02|
|2016-04-20 13:30:00|1e5b575e-19e7-44f...| click|adId_03|campaignId_01|
|2016-02-23 19:55:00|ed1b4467-8581-459...|  view|adId_06|campaignId_01|
|2016-10-12 09:51:00|7ebccca0-6039-47b...| click|adId_08|campaig

In [11]:
df.printSchema()

root
 |-- timestamp: timestamp (nullable = true)
 |-- user_id: string (nullable = true)
 |-- action: string (nullable = true)
 |-- adId: string (nullable = true)
 |-- campaignId: string (nullable = true)



## Resposta Atividade 1

Qual é o anúncio mais popular, ou seja, com mais eventos independente do action?

**Resposta:**  
adId_06

In [12]:
# ESCREVA SEU CÓDIGO AQUI

In [23]:
type(df.show())

+-------------------+--------------------+------+-------+-------------+
|          timestamp|             user_id|action|   adId|   campaignId|
+-------------------+--------------------+------+-------+-------------+
|2016-09-21 22:11:00|7c74953c-66cc-48b...| click|adId_09|campaignId_01|
|2016-06-25 18:29:00|676a083e-2f8e-4ff...|  view|adId_09|campaignId_02|
|2016-02-14 19:03:00|77158997-0dfa-48b...| click|adId_02|campaignId_02|
|2016-03-26 06:27:00|78aa2467-b502-413...| click|adId_07|campaignId_03|
|2016-01-02 04:57:00|fef9a98c-d73e-48e...|  view|adId_02|campaignId_02|
|2016-03-04 09:14:00|6ba65af9-4d83-456...|  view|adId_09|campaignId_01|
|2016-07-09 21:42:00|be3befb9-ee08-431...| click|adId_09|campaignId_01|
|2016-07-18 21:15:00|d37832ae-546f-4b9...| click|adId_05|campaignId_02|
|2016-04-20 13:30:00|1e5b575e-19e7-44f...| click|adId_03|campaignId_01|
|2016-02-23 19:55:00|ed1b4467-8581-459...|  view|adId_06|campaignId_01|
|2016-10-12 09:51:00|7ebccca0-6039-47b...| click|adId_08|campaig

NoneType

In [22]:
type(df.take(1)[0])

pyspark.sql.types.Row

In [24]:
df.show(1)[0]['adId']

+-------------------+--------------------+------+-------+-------------+
|          timestamp|             user_id|action|   adId|   campaignId|
+-------------------+--------------------+------+-------+-------------+
|2016-09-21 22:11:00|7c74953c-66cc-48b...| click|adId_09|campaignId_01|
+-------------------+--------------------+------+-------+-------------+
only showing top 1 row


TypeError: 'NoneType' object is not subscriptable

In [20]:
df.groupBy('adId')\
  .count()\
  .orderBy(F.desc('count'))\
  .take(1)[0]['adId']

'adId_06'

In [33]:
df.groupBy('adId')\
  .count()\
  .orderBy(F.desc('count'))\
  .first()['adId']

'adId_06'

## Resposta Atividade 2

Quantos clicks gerou a campanha mais popular?

**Resposta:**  
63983 clicks

In [13]:
# ESCREVA SEU CÓDIGO AQUI

In [27]:
df_popular = df.groupBy('campaignId')\
  .count()\
  .orderBy(F.desc('count'))\
  .take(1)[0]['campaignId']
df_popular

'campaignId_02'

In [28]:
#preferi essa
df.where(df.campaignId == df_popular)\
  .where(df.action == 'click')\
  .count()

63983

In [30]:
df.where((F.col('campaignId') == df_popular) & (F.col('action') == "click"))\
  .count()

63983

## Resposta Atividade 3

Algum usuário só visualizou? Quantas actions foram enviadas por usuário que só visualizou?

**Resposta:**  
Sim, o usuário d99871cb-98b7-4ac5-97a5-b9a26c0f897b enviou apenas 1 action

In [14]:
# ESCREVA SEU CÓDIGO AQUI

In [48]:
df.groupBy('user_id', 'action')\
  .count()\
  .groupBy('user_id')\
  .pivot('action')\
  .sum('count')\
  .where(F.col('click').isNull())\
  .toPandas()

,user_id,click,view
0,d99871cb-98b7-4ac5-97a5-b9a26c0f897b,NaN,1


## Resposta Atividade 4

Dos 10411 usuários, quantos usuários clicam mais que visualizam?

**Resposta:**  
10078

In [15]:
# ESCREVA SEU CÓDIGO AQUI

In [60]:
df.groupBy('user_id', 'action')\
  .count()\
  .groupBy('user_id')\
  .pivot('action')\
  .sum('count')\
  .fillna(0)\
  .where(F.col('click') > F.col('view'))\
  .count()

10078

## Resposta Atividade 5

Calcule a quantidade de clicks por dia da semana e apresente o resultado em ordem decrescente de quantidade de clicks.

PS: Nesse exemplo, 0 é segunda-feira e 6 é domingo. Você pode usar outro padrão, contanto que os dias da semana estejam correspondentes a resposta abaixo

**Resposta**  
27918, 4  
25424, 5  
25028, 0  
25027, 3  
25020, 6  
24973, 1  
24915, 2

In [16]:
# ESCREVA SEU CÓDIGO AQUI

In [65]:
result = df.where(F.col('action') == "click")\
  .withColumn('day_of_week', F.dayofweek('timestamp'))\
  .groupBy('day_of_week')\
  .count()\
  .orderBy(F.desc('count'))

result.toPandas()


,day_of_week,count
0,6,27918
1,7,25424
2,2,25028
3,5,25027
4,1,25020
5,3,24973
6,4,24915


## Resposta Atividade 6

Algum anúncio teve um crescimento de clicks ao longo das 3 campanhas?

**Resposta:**  
Sim, o anúncio adId_07

In [17]:
# ESCREVA SEU CÓDIGO AQUI

In [75]:
result_df= df.where(F.col('action') == "click")\
  .groupBy('adId', 'campaignId')\
  .count()\
  .groupBy('adId')\
  .pivot('campaignId')\
  .sum('count')\
  .fillna(0)\
  .orderBy('adId')

result_df.show()

+-------+-------------+-------------+-------------+
|   adId|campaignId_01|campaignId_02|campaignId_03|
+-------+-------------+-------------+-------------+
|adId_01|         5156|         6452|         5361|
|adId_02|         5073|         7423|         6509|
|adId_03|         5719|         7046|         5197|
|adId_04|         5025|         7023|         6348|
|adId_05|         5518|         6815|         5818|
|adId_06|         5533|         7923|         7074|
|adId_07|         5287|         6909|         7081|
|adId_08|         5451|         6900|         5689|
|adId_09|         5529|         7492|         5857|
|adId_10|         5084|            0|         6013|
+-------+-------------+-------------+-------------+



In [76]:
result_df.where((F.col('campaignId_01') < F.col('campaignId_02')) & (F.col('campaignId_02') < F.col('campaignId_03')))\
  .show()

+-------+-------------+-------------+-------------+
|   adId|campaignId_01|campaignId_02|campaignId_03|
+-------+-------------+-------------+-------------+
|adId_07|         5287|         6909|         7081|
+-------+-------------+-------------+-------------+

